# Model Training Notebook v2 — Group 13

**Input :** `chicago_features_processed.parquet` (30 features, produced by Feature Engineering Pipeline)

**Improvements over Phase 2 baseline:**

| Area | Phase 2 Baseline | v2 (This notebook) |
|---|---|---|
| Features | 12 (raw sums) | **30** (normalized means, surge, trend, cross-tier) |
| Lag type | Raw sum | **Rolling mean** (scale-independent) |
| XGBoost | scale_pos_weight=10, uncalibrated | **Tuned weight + calibration + regularization** |
| Class imbalance | scale_pos_weight only | **SMOTE for Tier 1 + class_weight for others** |
| Tier 1 training | Full train set | **Separate calibration set (2024)** for XGBoost |

**Pipeline design:** Feature engineering is separate (already done).
This notebook only does: load → train → evaluate → save.


## Step 1: Imports & Config

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# ── Paths ──────────────────────────────────────────────────────────────────
FOLDER   = 'G:/My Drive/To be - AI Medical Engineer/Claude_app_Project/IT5006 Group Project Final'
DATA_FILE = os.path.join(FOLDER, 'chicago_features_processed.parquet')
SAVE_DIR  = os.path.join(FOLDER, 'Saved_Models')
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_COLS  = ['y_tier1', 'y_tier2', 'y_tier3', 'y_tier4']
THRESHOLDS   = [0.10, 0.20, 0.35, 0.35]   # same as Phase 2

# Models that require scaled features
NEEDS_SCALING = {'Logistic Regression': True, 'Random Forest': False,
                 'XGBoost': False, 'Neural Network': True}

print('All imports successful.')


## Step 2: Load Processed Dataset & Split

In [ ]:
print('Loading processed dataset...')
df = pd.read_parquet(DATA_FILE)
print(f'  Shape   : {df.shape}')
print(f'  Columns : {df.columns.tolist()}')

# Load feature list saved by feature engineering pipeline
FEATURES = joblib.load(os.path.join(SAVE_DIR, 'feature_names_v2.joblib'))
print(f'  Features: {len(FEATURES)}')

# ── Train / Validation split ──────────────────────────────────────────────
train_df = df[df['split'] == 'train'].sort_values('Date').reset_index(drop=True)
val_df   = df[df['split'] == 'validation'].sort_values('Date').reset_index(drop=True)

# ── XGBoost calibration set: hold out 2024 from training ─────────────────
# Train XGBoost on 2015-2023, calibrate on 2024, evaluate on 2025
xgb_train_df = train_df[train_df['Date'].dt.year <= 2023]
xgb_calib_df = train_df[train_df['Date'].dt.year == 2024]

X_train      = train_df[FEATURES]
X_val        = val_df[FEATURES]
X_xgb_train  = xgb_train_df[FEATURES]
X_xgb_calib  = xgb_calib_df[FEATURES]

print(f'\nSplit summary:')
print(f'  Full train (2015-2024) : {len(train_df):>10,} rows')
print(f'  XGB train  (2015-2023) : {len(xgb_train_df):>10,} rows')
print(f'  XGB calib  (2024)      : {len(xgb_calib_df):>10,} rows')
print(f'  Validation (2025)      : {len(val_df):>10,} rows')
print()
print('Class balance:')
for col in TARGET_COLS:
    tr = train_df[col].mean()
    vl = val_df[col].mean()
    print(f'  {col}: train={tr:.1%}  val={vl:.1%}')


## Step 3: StandardScaler
Fit **only on training data** to prevent leakage.
Applied to Logistic Regression and Neural Network only.


In [ ]:
scaler = StandardScaler()
X_train_scaled     = scaler.fit_transform(X_train)       # fit + transform on train
X_val_scaled       = scaler.transform(X_val)             # transform only on val
X_xgb_calib_scaled = scaler.transform(X_xgb_calib)      # for calibration set

# Save scaler
joblib.dump(scaler, os.path.join(SAVE_DIR, 'data_scaler_v2.joblib'))
print(f'Scaler fitted on {len(X_train):,} training rows.')
print(f'Saved: data_scaler_v2.joblib')


## Step 4: Model Definitions

**Key changes from Phase 2:**
- **XGBoost**: `scale_pos_weight` computed from actual class ratio (not fixed at 10)
  + regularization (`max_depth=4`, `reg_alpha`, `reg_lambda`, `subsample`) to reduce overfitting
  + probability calibration (isotonic regression on 2024 holdout)
- **Random Forest**: increased to 200 trees, `min_samples_leaf=5`
- **Neural Network**: deeper `(128, 64, 32)`, longer patience
- **SMOTE**: applied to Tier 1 training data only (1.3% positive rate → severe imbalance)


In [ ]:
def build_models(pos_rate_per_tier):
    """Build model definitions. pos_rate_per_tier = list of 4 positive rates."""
    models = {}

    # ── Logistic Regression ────────────────────────────────────────────────
    models['Logistic Regression'] = [
        LogisticRegression(class_weight='balanced', max_iter=1000,
                           solver='saga', C=1.0, random_state=42)
        for _ in range(4)
    ]

    # ── Random Forest ──────────────────────────────────────────────────────
    models['Random Forest'] = [
        RandomForestClassifier(n_estimators=200, class_weight='balanced',
                               max_depth=10, min_samples_leaf=5,
                               n_jobs=-1, random_state=42)
        for _ in range(4)
    ]

    # ── XGBoost — scale_pos_weight from data, with regularization ──────────
    xgb_list = []
    for rate in pos_rate_per_tier:
        neg_rate = 1.0 - rate
        spw      = neg_rate / rate if rate > 0 else 1.0   # neg/pos ratio
        try:
            xgb = XGBClassifier(
                scale_pos_weight = round(spw, 2),
                max_depth        = 4,       # reduced from 6 → less overfitting
                n_estimators     = 300,
                learning_rate    = 0.05,
                reg_alpha        = 0.1,     # L1
                reg_lambda       = 1.0,     # L2
                subsample        = 0.8,
                colsample_bytree = 0.8,
                min_child_weight = 5,
                tree_method      = 'hist',
                device           = 'cuda',  # GPU
                eval_metric      = 'auc',
                random_state     = 42,
                verbosity        = 0
            )
        except Exception:
            xgb = XGBClassifier(
                scale_pos_weight = round(spw, 2),
                max_depth=4, n_estimators=300, learning_rate=0.05,
                reg_alpha=0.1, reg_lambda=1.0, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=5,
                tree_method='hist', eval_metric='auc',
                random_state=42, verbosity=0
            )
        xgb_list.append(xgb)
    models['XGBoost'] = xgb_list

    # ── Neural Network ─────────────────────────────────────────────────────
    models['Neural Network'] = [
        MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=300,
                      early_stopping=True, validation_fraction=0.1,
                      n_iter_no_change=20, random_state=42)
        for _ in range(4)
    ]

    return models

# Compute positive rates from training data
pos_rates = [train_df[col].mean() for col in TARGET_COLS]
print('Positive rates (training):')
for col, rate in zip(TARGET_COLS, pos_rates):
    neg_pos_ratio = (1 - rate) / rate
    print(f'  {col}: {rate:.3f} → scale_pos_weight = {neg_pos_ratio:.1f}')


## Step 5: Time-Series Cross-Validation
3-fold `TimeSeriesSplit` on training data to estimate generalisation performance.
Reports mean AUC-ROC per model × tier.


In [ ]:
tscv       = TimeSeriesSplit(n_splits=3)
cv_results = []
models_def = build_models(pos_rates)

print('Running Time-Series CV (3 folds)...')
print('-' * 55)

for model_name, estimators in models_def.items():
    use_scaled = NEEDS_SCALING[model_name]
    X_cv = X_train_scaled if use_scaled else X_train.values

    for i, col in enumerate(TARGET_COLS):
        y_cv  = train_df[col].values
        aucs  = []

        for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_cv)):
            X_f, y_f = X_cv[tr_idx], y_cv[tr_idx]
            X_v, y_v = X_cv[vl_idx], y_cv[vl_idx]

            # SMOTE only for Tier 1 (severe imbalance)
            if col == 'y_tier1' and y_f.sum() > 10:
                try:
                    smote = SMOTE(sampling_strategy=0.15, random_state=42, k_neighbors=3)
                    X_f, y_f = smote.fit_resample(X_f, y_f)
                except Exception:
                    pass  # if too few samples, skip SMOTE

            from sklearn.base import clone
            clf = clone(estimators[i])
            clf.fit(X_f, y_f)
            prob = clf.predict_proba(X_v)[:, 1]

            try:
                aucs.append(roc_auc_score(y_v, prob))
            except ValueError:
                aucs.append(np.nan)

        mean_auc = np.nanmean(aucs)
        cv_results.append({'Model': model_name, 'Tier': col,
                           'CV_AUC_mean': round(mean_auc, 4),
                           'CV_AUC_folds': [round(a, 4) for a in aucs]})
        print(f'  {model_name:>20} | {col} | AUC = {mean_auc:.4f}')

cv_df = pd.DataFrame(cv_results)
print('\nCV complete.')


## Step 6: Final Model Training
Train each model on **full training set (2015–2024)**.

**XGBoost exception:** trained on 2015–2023, then probability-calibrated on 2024
using isotonic regression → fixes miscalibrated scores on unseen data.


In [ ]:
from sklearn.base import clone

models_def      = build_models(pos_rates)
trained_models  = {}   # stores final fitted estimators

print('Training final models...')
print('=' * 55)

for model_name, estimators in models_def.items():
    use_scaled   = NEEDS_SCALING[model_name]
    is_xgb       = (model_name == 'XGBoost')
    trained_list = []

    for i, col in enumerate(TARGET_COLS):
        # ── Choose training set ───────────────────────────────────────────
        if is_xgb:
            # XGBoost: train on 2015-2023, calibrate on 2024
            X_fit = X_xgb_train.values
            y_fit = xgb_train_df[col].values
            X_cal = X_xgb_calib.values
            y_cal = xgb_calib_df[col].values
        else:
            X_fit = X_train_scaled if use_scaled else X_train.values
            y_fit = train_df[col].values

        # ── SMOTE for Tier 1 only ─────────────────────────────────────────
        if col == 'y_tier1' and y_fit.sum() > 10:
            try:
                smote = SMOTE(sampling_strategy=0.15, random_state=42, k_neighbors=3)
                X_fit, y_fit = smote.fit_resample(X_fit, y_fit)
                print(f'  [{model_name}] Tier 1 SMOTE: {y_fit.sum()} positives after resampling')
            except Exception as e:
                print(f'  [{model_name}] Tier 1 SMOTE skipped: {e}')

        # ── Train ─────────────────────────────────────────────────────────
        clf = clone(estimators[i])
        clf.fit(X_fit, y_fit)

        # ── XGBoost: probability calibration on 2024 holdout ─────────────
        if is_xgb:
            calibrated_clf = CalibratedClassifierCV(
                clf, method='isotonic', cv='prefit'
            )
            calibrated_clf.fit(X_cal, y_cal)
            trained_list.append(calibrated_clf)
            print(f'  XGBoost {col}: trained + calibrated')
        else:
            trained_list.append(clf)
            print(f'  {model_name} {col}: trained')

    trained_models[model_name] = trained_list

print('\nAll models trained.')


## Step 7: Evaluation on 2025 Validation Set

In [ ]:
eval_results  = []
all_proba     = {}   # store probabilities for threshold analysis

for model_name, estimators in trained_models.items():
    use_scaled = NEEDS_SCALING[model_name]
    X_input    = X_val_scaled if use_scaled else X_val.values
    prob_matrix = []

    for i, col in enumerate(TARGET_COLS):
        clf  = estimators[i]
        prob = clf.predict_proba(X_input)[:, 1]
        pred = (prob >= THRESHOLDS[i]).astype(int)
        prob_matrix.append(prob)

        y_true = val_df[col].values
        try:
            auc = roc_auc_score(y_true, prob)
        except ValueError:
            auc = np.nan

        eval_results.append({
            'Model'     : model_name,
            'Tier'      : col,
            'Precision' : round(precision_score(y_true, pred, zero_division=0), 4),
            'Recall'    : round(recall_score(y_true,    pred, zero_division=0), 4),
            'F1'        : round(f1_score(y_true,        pred, zero_division=0), 4),
            'AUC'       : round(auc, 4) if not np.isnan(auc) else np.nan
        })

    all_proba[model_name] = np.array(prob_matrix).T

eval_df = pd.DataFrame(eval_results)
print('=== VALIDATION RESULTS (2025) ===')
print(eval_df.to_string(index=False))


## Step 8: Compare v2 vs Phase 2 Baseline

In [ ]:
# Phase 2 baseline results (from original notebook)
baseline = [
    # Model,               Tier,       Prec,   Rec,    F1,     AUC
    ('Logistic Regression','y_tier1',  0.2211, 0.3931, 0.2830, 0.7329),
    ('Logistic Regression','y_tier2',  0.4764, 0.9991, 0.6452, 0.7222),
    ('Logistic Regression','y_tier3',  0.7795, 0.9965, 0.8748, 0.7166),
    ('Logistic Regression','y_tier4',  0.5617, 0.4531, 0.5016, 0.7483),
    ('Random Forest',       'y_tier1',  0.0419, 0.0162, 0.0234, 0.6134),
    ('Random Forest',       'y_tier2',  0.4894, 0.9697, 0.6505, 0.6176),
    ('Random Forest',       'y_tier3',  0.7818, 0.9809, 0.8701, 0.6760),
    ('Random Forest',       'y_tier4',  0.3904, 0.1380, 0.2039, 0.6090),
    ('XGBoost',             'y_tier1',  0.0327, 0.2711, 0.0583, 0.3244),
    ('XGBoost',             'y_tier2',  0.4600, 0.9345, 0.6166, 0.5606),
    ('XGBoost',             'y_tier3',  0.7769, 0.9980, 0.8737, 0.7023),
    ('XGBoost',             'y_tier4',  0.2347, 0.6969, 0.3511, 0.5272),
    ('Neural Network',      'y_tier1',  0.0816, 0.0034, 0.0065, 0.3939),
    ('Neural Network',      'y_tier2',  0.4833, 0.9878, 0.6490, 0.7218),
    ('Neural Network',      'y_tier3',  0.7801, 0.9957, 0.8749, 0.7237),
    ('Neural Network',      'y_tier4',  0.5602, 0.3382, 0.4218, 0.7447),
]
base_df = pd.DataFrame(baseline, columns=['Model','Tier','Precision','Recall','F1','AUC'])

# Merge and compute delta
comp = eval_df.merge(base_df, on=['Model','Tier'], suffixes=('_v2','_base'))
comp['F1_delta']  = (comp['F1_v2']  - comp['F1_base']).round(4)
comp['AUC_delta'] = (comp['AUC_v2'] - comp['AUC_base']).round(4)

print('=== F1 and AUC improvement (v2 - baseline) ===')
print(comp[['Model','Tier','F1_base','F1_v2','F1_delta','AUC_base','AUC_v2','AUC_delta']].to_string(index=False))
print()
print(f'Mean F1  improvement : {comp["F1_delta"].mean():+.4f}')
print(f'Mean AUC improvement : {comp["AUC_delta"].mean():+.4f}')


## Step 9: Feature Importance (XGBoost — All Tiers)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

tier_labels = ['Tier 1 - Lethal', 'Tier 2 - Personal Violence',
               'Tier 3 - Property', 'Tier 4 - Public Order']

for i, (ax, label) in enumerate(zip(axes, tier_labels)):
    xgb_cal  = trained_models['XGBoost'][i]
    # CalibratedClassifierCV wraps the base estimator
    base_xgb = xgb_cal.calibrated_classifiers_[0].estimator
    importances = base_xgb.feature_importances_

    feat_df = pd.DataFrame({'Feature': FEATURES, 'Importance': importances})
    feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)

    ax.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
    ax.set_title(f'XGBoost Feature Importance\n{label}', fontsize=11)
    ax.set_xlabel('Importance')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FOLDER, 'feature_importance_v2.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance plot saved.')


## Step 10: Threshold Sensitivity (Tier 2 — Personal Violence)

In [ ]:
threshold_grid = [0.20, 0.25, 0.35, 0.45, 0.55, 0.65]
y_true_t2      = val_df['y_tier2'].values
base_rate      = y_true_t2.mean()

print(f'Base rate (Tier 2, 2025 val): {base_rate:.3f}')
print()

for model_name, prob_matrix in all_proba.items():
    print(f'--- {model_name} ---')
    print(f'  {"Threshold":<10} {"Precision":<12} {"Recall":<10} {"F1":<8} {"Lift":<8} {"Days Flagged"}')
    print('  ' + '-' * 62)
    for t in threshold_grid:
        pred  = (prob_matrix[:, 1] >= t).astype(int)
        flags = pred.sum()
        if flags > 0:
            prec  = precision_score(y_true_t2, pred, zero_division=0)
            rec   = recall_score(y_true_t2, pred, zero_division=0)
            f1    = f1_score(y_true_t2, pred, zero_division=0)
            lift  = prec / base_rate
        else:
            prec = rec = f1 = lift = 0
        print(f'  {t:<10.2f} {prec:<12.1%} {rec:<10.1%} {f1:<8.4f} {lift:<8.2f}x {flags}')
    print()


## Step 11: Weekday vs Weekend Accuracy (Tier 2)

In [ ]:
val_df_eval = val_df.copy()
val_df_eval['is_weekend'] = val_df_eval['Date'].dt.dayofweek >= 5
weekend_mask = val_df_eval['is_weekend'].values
weekday_mask = ~weekend_mask

print(f'  {"Model":<22} {"Weekday Acc":>12} {"Weekend Acc":>12} {"Diff":>8}')
print('  ' + '-' * 58)
for model_name, prob_matrix in all_proba.items():
    pred_t2 = (prob_matrix[:, 1] >= 0.20).astype(int)
    y_true  = val_df['y_tier2'].values
    wd_acc  = (pred_t2[weekday_mask] == y_true[weekday_mask]).mean()
    we_acc  = (pred_t2[weekend_mask] == y_true[weekend_mask]).mean()
    print(f'  {model_name:<22} {wd_acc:>11.2%} {we_acc:>11.2%} {we_acc-wd_acc:>+7.2%}')


## Step 12: Save All Artifacts

In [ ]:
# Save trained models
joblib.dump(trained_models, os.path.join(SAVE_DIR, 'crime_models_v2.joblib'))
# Save evaluation results
joblib.dump(eval_df,         os.path.join(SAVE_DIR, 'eval_results_v2.joblib'))
joblib.dump(cv_df,           os.path.join(SAVE_DIR, 'cv_results_v2.joblib'))

print('Saved artifacts:')
artifacts = [
    ('crime_models_v2.joblib',    'All 4 trained models (16 classifiers + calibrated XGBoost)'),
    ('data_scaler_v2.joblib',     'StandardScaler fitted on 2015-2024 training data'),
    ('feature_names_v2.joblib',   '30 feature names (from feature engineering pipeline)'),
    ('composite_weights.joblib',  'Composite risk weights per target tier'),
    ('eval_results_v2.joblib',    'Validation (2025) performance metrics'),
    ('cv_results_v2.joblib',      'Time-series CV AUC results'),
]
for fname, desc in artifacts:
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f'  {fname:<38} {size:>8.1f} KB  — {desc}')


## Step 13: Final Summary

In [ ]:
print('=' * 70)
print('  MODEL TRAINING v2 — COMPLETE SUMMARY')
print('=' * 70)
print(f'  Features          : 30 (vs 12 in Phase 2 baseline)')
print(f'  Training records  : {len(train_df):,} (2015-2024)')
print(f'  Validation records: {len(val_df):,} (2025)')
print()
print('  CV Performance (mean AUC across 3 time-series folds):')
pivot_cv = cv_df.pivot(index='Model', columns='Tier', values='CV_AUC_mean')
print(pivot_cv.round(4).to_string())
print()
print('  Validation Performance (2025 — AUC):')
pivot_val = eval_df.pivot(index='Model', columns='Tier', values='AUC')
print(pivot_val.round(4).to_string())
print()
print('  Validation Performance (2025 — F1):')
pivot_f1 = eval_df.pivot(index='Model', columns='Tier', values='F1')
print(pivot_f1.round(4).to_string())
print()
print('  Improvement vs Phase 2 baseline:')
print(f'    Mean F1  change: {comp["F1_delta"].mean():+.4f}')
print(f'    Mean AUC change: {comp["AUC_delta"].mean():+.4f}')
print()
print('  Next step: Update NIBRS testing notebook to use crime_models_v2.joblib')
print('             and feature_names_v2.joblib')
